<a href="https://colab.research.google.com/github/Aham0803/PySpark/blob/main/aggregation_and_grouping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [67]:
import pyspark

In [68]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [69]:
spark = SparkSession.builder.appName("GroupByDemo").getOrCreate()

In [70]:
data =[
    ("IT" , "Gaurav" , 100000),
    ("Sales" , "Aman" , 550000),
    ("IT" , "Saurabh" , 800000),
    ("Marketing" ,"Dev" , 500000),
    ("Sales","Amit" , 500000)
]
schema =["department" , "name" , "salary"]
df = spark.createDataFrame  (data , schema)

In [71]:
df.show()

+----------+-------+------+
|department|   name|salary|
+----------+-------+------+
|        IT| Gaurav|100000|
|     Sales|   Aman|550000|
|        IT|Saurabh|800000|
| Marketing|    Dev|500000|
|     Sales|   Amit|500000|
+----------+-------+------+



simple aggregate and grupby

budget for each department

In [72]:
df.groupBy("department").sum("salary").show()

+----------+-----------+
|department|sum(salary)|
+----------+-----------+
|     Sales|    1050000|
|        IT|     900000|
| Marketing|     500000|
+----------+-----------+



avg salary of employee in each department

In [73]:
df.groupBy("department").avg("salary").show()

+----------+-----------+
|department|avg(salary)|
+----------+-----------+
|     Sales|   525000.0|
|        IT|   450000.0|
| Marketing|   500000.0|
+----------+-----------+



for adv aggregation we can use .agg() function . It allows to run multiple calculations as once and rename the output for cleaner look

In [74]:
df_summary = df.groupBy("department").agg(
    F.sum("salary").alias("TotalSalary"),
    F.avg("salary").alias("AvgSalary"),
    F.count("name").alias("TotalEmp"),
    F.max("salary").alias("MaxSalary")
)

In [75]:
df_summary.show()

+----------+-----------+---------+--------+---------+
|department|TotalSalary|AvgSalary|TotalEmp|MaxSalary|
+----------+-----------+---------+--------+---------+
|     Sales|    1050000| 525000.0|       2|   550000|
|        IT|     900000| 450000.0|       2|   800000|
| Marketing|     500000| 500000.0|       1|   500000|
+----------+-----------+---------+--------+---------+



In [76]:
df_summary.filter(F.col("TotalSalary")>500000).show()

+----------+-----------+---------+--------+---------+
|department|TotalSalary|AvgSalary|TotalEmp|MaxSalary|
+----------+-----------+---------+--------+---------+
|     Sales|    1050000| 525000.0|       2|   550000|
|        IT|     900000| 450000.0|       2|   800000|
+----------+-----------+---------+--------+---------+



In [77]:
df.groupBy("department" , "salary").count().show()

+----------+------+-----+
|department|salary|count|
+----------+------+-----+
|     Sales|550000|    1|
|        IT|100000|    1|
| Marketing|500000|    1|
|     Sales|500000|    1|
|        IT|800000|    1|
+----------+------+-----+



#condtional aggregation with pyspark

here we will use groupby method along with aggregate functions like sum , count etc with f.when() methods

In [78]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [79]:
spark = SparkSession.builder.appName("conditionalAggDemo").getOrCreate()

this data stores about Online or in store sales. amount could be positive(buy) or negative(return)

In [80]:
data =[
    ("Store_A", "Online", 150),
    ("Store_A" , "In-Store" , 50),
    ("Store_A" , "Online" , 300),
    ("Store_B" , "In-Store" , 400),
    ("Store_B" , "Online" , -100),
    ("Store_B" , "In-Store" , 200),
]
columns =["Store" , "Type" ,"Amount"]

In [81]:
df = spark.createDataFrame(data , columns)
df.show()

+-------+--------+------+
|  Store|    Type|Amount|
+-------+--------+------+
|Store_A|  Online|   150|
|Store_A|In-Store|    50|
|Store_A|  Online|   300|
|Store_B|In-Store|   400|
|Store_B|  Online|  -100|
|Store_B|In-Store|   200|
+-------+--------+------+



to use aggregate function only on specific rows , we use  F.when(condtion , value)

In [87]:
df_conditional = df.groupBy("Store").agg(
    F.sum(F.when(F.col("Amount")>0 , F.col("Amount"))).alias("TotalSales"),
    F.sum(F.when(F.col("Amount")<0 , F.col("Amount"))).alias("TotalReturns"),
    F.count(F.when(F.col("Type")== "Online" , 1)).alias("OnlineTransactionsCount")
)

In [88]:
df_conditional.show()

+-------+----------+------------+-----------------------+
|  Store|TotalSales|TotalReturns|OnlineTransactionsCount|
+-------+----------+------------+-----------------------+
|Store_A|       500|        NULL|                      2|
|Store_B|       600|        -100|                      1|
+-------+----------+------------+-----------------------+



conditional avg and persentage

In [89]:
df_analysis = df.groupBy("Store").agg(
    F.avg(F.when(F.col("Type") == "Online" , F.col("Amount"))).alias("AvegrageOnlineAmount"), ## mean amount for Online orders only
    (F.count(F.when(F.col("Type") == "In-Store",1)) / F.count("*")*100).alias("In-StorePercentage")  ## conditional percentge : (In-Store transaction) /(total_transaction)*100
)

In [90]:
df_analysis.show()

+-------+--------------------+------------------+
|  Store|AvegrageOnlineAmount|In-StorePercentage|
+-------+--------------------+------------------+
|Store_A|               225.0| 33.33333333333333|
|Store_B|              -100.0| 66.66666666666666|
+-------+--------------------+------------------+



use .nafill method(data imputaion

In [91]:
df_Cleaned = df_conditional.na.fill(0,["TotalReturns"])
df_Cleaned.show()

+-------+----------+------------+-----------------------+
|  Store|TotalSales|TotalReturns|OnlineTransactionsCount|
+-------+----------+------------+-----------------------+
|Store_A|       500|           0|                      2|
|Store_B|       600|        -100|                      1|
+-------+----------+------------+-----------------------+

